In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_reviews.csv")

print(df.shape)

(219510, 5)


In [2]:
print(df["ProductId"].value_counts().describe())

count    17538.000000
mean        12.516250
std         37.540499
min          1.000000
25%          2.000000
50%          3.000000
75%          7.000000
max        564.000000
Name: count, dtype: float64


In [3]:
product_stats = df.groupby("ProductId").agg(
    avg_rating=("Score", "mean"),
    review_count=("Score", "count")
)

product_stats.head()

,avg_rating,review_count
ProductId,,
0006641040,4.000000,3
7310172001,4.751445,173
7310172101,4.751445,173
B00002N8SM,1.000000,2
B00004CI84,4.037037,27


In [4]:
popular_products = product_stats[
    product_stats["review_count"] >= 5
]

In [5]:
popular_products = popular_products.sort_values(
    by="avg_rating",
    ascending=False
)

popular_products.head(10)

,avg_rating,review_count
ProductId,,
B001EO5Y7K,5.0,5
B001LQRKC8,5.0,6
B001NC8HS6,5.0,6
B001M22LX4,5.0,5
B001M0AK5M,5.0,7
B001M09BOS,5.0,7
B001M072X0,5.0,5
B001LQTJ3Q,5.0,5
B001LQRFLE,5.0,5


In [7]:
product_stats.sort_values(
    by="review_count",
    ascending=False
).head(20)

,avg_rating,review_count
ProductId,,
B007M832YY,4.310284,564
B0026KPDG8,4.310284,564
B0026KNQSA,4.310284,564
B006HYLW32,4.310284,564
B0013NUGDE,4.310284,564
B001RVFERK,4.310284,564
B007M83302,4.310284,564
B001RVFEP2,4.310284,564
B000VK8AVK,4.310284,564


In [8]:
global_avg = df["Score"].mean()

print(global_avg)

4.197635643023097


In [9]:
m = 20

popular_products["weighted_score"] = (
    (popular_products["review_count"] * popular_products["avg_rating"])
    + (m * global_avg)
) / (popular_products["review_count"] + m)

In [10]:
popular_products = popular_products.sort_values(
    by="weighted_score",
    ascending=False
)

popular_products.head(10)

,avg_rating,review_count,weighted_score
ProductId,,,
B001EO5Q64,4.846154,104,4.741554
B001CWX7EG,4.789116,147,4.718280
B001CWV4RS,4.789116,147,4.718280
B001CWSKFC,4.789116,147,4.718280
B000EVG8HY,4.789116,147,4.718280
B000EVIDWW,4.789116,147,4.718280
B000EVG8FQ,4.789116,147,4.718280
B0030VBQ5Y,4.784483,116,4.698182
B0030VBPN2,4.784483,116,4.698182


In [11]:
top_products = popular_products.head(20)

top_products

,avg_rating,review_count,weighted_score
ProductId,,,
B001EO5Q64,4.846154,104,4.741554
B001CWX7EG,4.789116,147,4.718280
B001CWV4RS,4.789116,147,4.718280
B001CWSKFC,4.789116,147,4.718280
B000EVG8HY,4.789116,147,4.718280
B000EVIDWW,4.789116,147,4.718280
B000EVG8FQ,4.789116,147,4.718280
B0030VBQ5Y,4.784483,116,4.698182
B0030VBPN2,4.784483,116,4.698182


In [13]:
top_products.to_csv(
    "../data//popularity_recommendations.csv"
)

In [14]:
user_product_matrix = df.pivot_table(
    index="UserId",
    columns="ProductId",
    values="Score"
)

In [16]:

print(user_product_matrix.shape)

(23260, 17538)


In [17]:
product_counts = df["ProductId"].value_counts()

popular_products = product_counts[
    product_counts >= 50
].index

df_cf = df[
    df["ProductId"].isin(popular_products)
]

print(df_cf.shape)
print(df_cf["ProductId"].nunique())
print(df_cf["UserId"].nunique())

(132643, 5)
980
15672


In [18]:
user_product_matrix_cf = df_cf.pivot_table(
    index="UserId",
    columns="ProductId",
    values="Score"
)

print(user_product_matrix_cf.shape)

(15672, 980)


In [19]:
user_product_matrix_cf = user_product_matrix_cf.fillna(0)

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity = cosine_similarity(
    user_product_matrix_cf.T
)

print(item_similarity.shape)

(980, 980)


In [24]:
import pickle
product_ids = user_product_matrix_cf.columns

with open("../data/models/product_ids.pkl", "wb") as f:
    pickle.dump(product_ids, f)
    
with open("../data/models/item_similarity.pkl", "wb") as f:
    pickle.dump(item_similarity, f)

In [43]:
product_to_index = {
    product: idx
    for idx, product in enumerate(product_ids)
}

def recommend_collaborative(product_id, top_n=5):
    
    idx = product_to_index[product_id]

    similarity_scores = item_similarity[idx]

    similar_indices = similarity_scores.argsort()[::-1][1:top_n+1]

    return product_ids[similar_indices]



In [44]:
print(product_ids[:10])

Index(['0006641040', '7310172001', '7310172101', 'B00002N8SM', 'B00004CI84',
       'B00004CXX9', 'B00004RAMV', 'B00004RAMX', 'B00004RAMY', 'B00004RBDU'],
      dtype='object', name='ProductId')


In [45]:
recommend_collaborative("B000084DWM")

Index(['B0000SXEN2', 'B000084DWM', 'B00008433V', 'B000634IC2', 'B0002ASMT4'], dtype='object', name='ProductId')

In [28]:
df[["ProductId", "Summary"]].head(20)

,ProductId,Summary
0,B001GVISJM,fresh and greasy!
1,B001GVISJM,Strawberry Twizzlers - Yummy
2,B001GVISJM,GREAT SWEET CANDY!
3,B001EO5QW8,Best of the Instant Oatmeals
4,B001EO5QW8,Wife's favorite Breakfast
5,B001EO5QW8,Why wouldn't you buy oatmeal from Mcanns? Tast...
6,B001EO5QW8,Good Hot Breakfast
7,B001EO5QW8,Great taste and convenience
8,B001EO5QW8,Hearty Oatmeal
9,B001EO5QW8,good


In [29]:
df["combined_text"] = (
    df["Summary"].fillna("") + " " +
    df["Text"].fillna("")
)

In [30]:
product_text = df.groupby("ProductId")["combined_text"].apply(
    " ".join
)

print(product_text.shape)

(17538,)


In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [33]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(product_text)

print(tfidf_matrix.shape)

(17538, 5000)


In [34]:
from sklearn.metrics.pairwise import cosine_similarity

content_similarity = cosine_similarity(
    tfidf_matrix
)

In [35]:
print(content_similarity.shape)

(17538, 17538)


In [36]:
import pickle

with open("../data/models/content_similarity.pkl", "wb") as f:
    pickle.dump(content_similarity, f)
    
with open("../data/models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)
    
with open("../data/models/content_product_ids.pkl", "wb") as f:
    pickle.dump(product_text.index, f)

In [37]:
product_ids = product_text.index

product_to_index = {
    product: idx
    for idx, product in enumerate(product_ids)
}

In [38]:
def recommend_content(product_id, top_n=5):
    
    idx = product_to_index[product_id]

    similarity_scores = content_similarity[idx]

    similar_indices = similarity_scores.argsort()[::-1][1:top_n+1]

    return product_ids[similar_indices]

In [40]:
print(product_ids[:5])

Index(['0006641040', '7310172001', '7310172101', 'B00002N8SM', 'B00004CI84'], dtype='object', name='ProductId')


In [46]:
recommend_content("B000084DWM")

Index(['B000QSN7P6', 'B001VIYCK4', 'B000084DWM', 'B009B87SAC', 'B003MWBFXY'], dtype='object', name='ProductId')